# Module 5 - Class 5: Anomaly Detection + Association Rules
**Khamidullokhon Abduvokhidov**

Upload `creditcard.csv` before running Activity 3.

In [ ]:
# Load fraud data, scale Amount and Time, and preserve the class balance in the split.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, ConfusionMatrixDisplay
df = pd.read_csv('creditcard.csv')
print(df.shape); print(df['Class'].value_counts())
X, y = df.drop(columns='Class'), df['Class']
X[['Time','Amount']] = StandardScaler().fit_transform(X[['Time','Amount']])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

In [ ]:
# Compare Isolation Forest settings and inspect the actual-rate model.
rows = []
for contamination in [0.00173, 0.005, 0.01]:
    model = IsolationForest(contamination=contamination, random_state=42).fit(X_train)
    pred = (model.predict(X_test) == -1).astype(int)
    rows.append([contamination, precision_score(y_test,pred), recall_score(y_test,pred), f1_score(y_test,pred)])
results = pd.DataFrame(rows, columns=['Contamination','Precision','Recall','F1'])
display(results)
ConfusionMatrixDisplay.from_predictions(y_test, pred); plt.title('Isolation Forest Confusion Matrix'); plt.show()
scores = model.decision_function(X_test)
for k in [10,50,100,200,500]: print(k, 'precision@k:', y_test.iloc[np.argsort(scores)[:k]].mean())
plt.plot([10,50,100,200,500], [y_test.iloc[np.argsort(scores)[:k]].mean() for k in [10,50,100,200,500]], 'o-'); plt.xlabel('k'); plt.ylabel('Precision@k'); plt.show()

In [ ]:
# Compute manual support, confidence, and lift for the ten transactions.
transactions = [['Bread','Butter','Milk'],['Bread','Butter'],['Bread','Milk','Tea'],['Butter','Milk','Tea'],['Bread','Butter','Milk','Tea'],['Bread','Tea'],['Milk','Tea'],['Bread','Butter','Milk'],['Bread','Milk'],['Butter','Tea']]
print('Supports: Bread=.70, Butter=.60, Milk=.70, Tea=.60')
print('Pair supports: Bread+Butter=.40, Bread+Milk=.50, Milk+Tea=.40')
print('Bread => Butter: confidence=.5714, lift=.9524')
print('Butter => Bread: confidence=.6667, lift=.9524')

In [ ]:
# Verify the association rules in code and translate strong rules into telecom offers.
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
encoded = TransactionEncoder().fit(transactions).transform(transactions)
basket = pd.DataFrame(encoded, columns=TransactionEncoder().fit(transactions).columns_)
frequent_items = apriori(basket, min_support=.3, use_colnames=True)
rules = association_rules(frequent_items, metric='confidence', min_threshold=.5)
display(rules[['antecedents','consequents','support','confidence','lift']].sort_values('lift', ascending=False))

## Telecom recommendations
If a customer buys home internet and a TV package together more often than expected, offer a discounted bundle at checkout. If mobile service predicts a home-internet purchase, target mobile-only customers with a relevant internet offer. Use lift, not confidence alone, to prioritize rules that show a genuine positive association.